# Chapter 05 - Getting Started with pandas

-----

In [4]:
%run "Ch00 - Basic Imports and Set ups.ipynb"

Pandas will be a major tool of interest throughout much of the rest of the book. It
contains data structures and data manipulation tools designed to make data cleaning
and analysis fast and easy in Python. pandas is often used in tandem with numerical
computing tools like NumPy and SciPy, analytical libraries like statsmodels and
scikit-learn, and data visualization libraries like matplotlib. pandas adopts significant
parts of NumPy’s idiomatic style of array-based computing, especially array-based
functions and a preference for data processing without for loops.

While pandas adopts many coding idioms from NumPy, the biggest difference is that
pandas is designed for working with tabular or heterogeneous data. NumPy, by contrast,
is best suited for working with homogeneous numerical array data.

Since becoming an open source project in 2010, pandas has matured into a quite
large library that’s applicable in a broad set of real-world use cases. The developer
community has grown to over 800 distinct contributors, who’ve been helping build
the project as they’ve used it to solve their day-to-day data problems.

In [ ]:
import pandas as pd
import numpy as np

## 5.1 Introduction to pandas Data Structures

-----

To get started with pandas, you will need to get comfortable with its two workhorse
data structures: Series and DataFrame. While they are not a universal solution for
every problem, they provide a solid, easy-to-use basis for most applications.

### Series

----

A Series is a one-dimensional array-like object containing a sequence of values (of
similar types to NumPy types) and an associated array of data labels, called its index.
The simplest Series is formed from only an array of data:

In [36]:
obj = pd.Series([4, 7, -5, 3])

In [37]:
obj

0    4
1    7
2   -5
3    3
dtype: int64

The string representation of a Series displayed interactively shows the index on the
left and the values on the right. Since we did not specify an index for the data, a
default one consisting of the integers 0 through N - 1 (where N is the length of the
data) is created. You can get the array representation and index object of the Series via
its values and index attributes, respectively:

In [38]:
obj.values

array([ 4,  7, -5,  3])

In [39]:
obj.index

RangeIndex(start=0, stop=4, step=1)

You can define custom idex values.

In [40]:
obj2 = pd.Series([4, 7, -5, 3], index=['d', 'b', 'a', 'c'])

In [41]:
obj2

d    4
b    7
a   -5
c    3
dtype: int64

In [42]:
obj2.index

Index(['d', 'b', 'a', 'c'], dtype='object')

Compared with NumPy arrays, you can use labels in the index when selecting single
values or a set of values:

In [43]:
obj2['a']

np.int64(-5)

In [44]:
obj2['d'] = 6

In [45]:
obj2['c', 'a', 'd']

KeyError: 'key of type tuple not found and not a MultiIndex'

In [46]:
obj2[['c', 'a', 'd']]

c    3
a   -5
d    6
dtype: int64

Using NumPy functions or NumPy-like operations, such as filtering with a boolean
array, scalar multiplication, or applying math functions, will preserve the index-value
link:

In [47]:
obj2 > 0

d     True
b     True
a    False
c     True
dtype: bool

In [48]:
obj2[obj2 > 0]

d    6
b    7
c    3
dtype: int64

In [49]:
obj2 * 2

d    12
b    14
a   -10
c     6
dtype: int64

In [50]:
np.exp(obj2)

d     403.428793
b    1096.633158
a       0.006738
c      20.085537
dtype: float64

Another way to think about a Series is as a fixed-length, ordered dict, as it is a mapping
of index values to data values. It can be used in many contexts where you might
use a dict:

In [51]:
'b' in obj2

True

In [52]:
'f' in obj2

False

Should you have data contained in a Python dict, you can create a Series from it by
passing the dict:

In [53]:
sdata = {'Ohio': 35000, 'Texas': 71000, 'Oregon': 16000, 'Utah': 5000}
obj3 = pd.Series(sdata)

In [54]:
obj3

Ohio      35000
Texas     71000
Oregon    16000
Utah       5000
dtype: int64

(See that data is ordered by index in the resulting Series.) 

When you are only passing a dict, the index in the resulting Series will have the dict’s
keys in sorted order. You can override this by passing the dict keys in the order you
want them to appear in the resulting Series:

In [55]:
states = ['California', 'Ohio', 'Oregon', 'Texas']

In [56]:
obj4 = pd.Series(sdata, index=states)

In [57]:
obj4

California        NaN
Ohio          35000.0
Oregon        16000.0
Texas         71000.0
dtype: float64

Here, three values found in sdata were placed in the appropriate locations, but since
no value for `'California'` was found, it appears as `NaN` (not a number), which is considered
in pandas to mark missing or NA values. Since `'Utah'` was not included in
states, it is excluded from the resulting object.

In [ ]:
pd.isnull(obj4)

In [ ]:
pd.notnull(obj4)

A useful Series feature for many applications is that it automatically aligns by index
label in arithmetic operations:

In [66]:
obj3

Ohio      35000
Texas     71000
Oregon    16000
Utah       5000
dtype: int64

In [67]:
obj4

California        NaN
Ohio          35000.0
Oregon        16000.0
Texas         71000.0
dtype: float64

In [68]:
obj3+obj4

California         NaN
Ohio           70000.0
Oregon         32000.0
Texas         142000.0
Utah               NaN
dtype: float64

It added values where indixes existing in both objects. Made it NaN for others

Both the Series object itself and its index have a name attribute, which integrates with
other key areas of pandas functionality:

In [69]:
obj4.name = 'population'
obj4.index.name = 'state'

In [70]:
obj4

state
California        NaN
Ohio          35000.0
Oregon        16000.0
Texas         71000.0
Name: population, dtype: float64

A Series’s index can be altered in-place by assignment:

In [71]:
obj

0    4
1    7
2   -5
3    3
dtype: int64

In [72]:
obj.index = ['Bob', 'Steve', 'Jeff', 'Ryan']

In [73]:
obj

Bob      4
Steve    7
Jeff    -5
Ryan     3
dtype: int64

----

### Data Frame

A DataFrame represents a rectangular table of data and contains an ordered collection
of columns, each of which can be a different value type (numeric, string,
boolean, etc.). The DataFrame has both a row and column index; it can be thought of
as a dict of Series all sharing the same index. Under the hood, the data is stored as one
or more two-dimensional blocks rather than a list, dict, or some other collection of
one-dimensional arrays. The exact details of DataFrame’s internals are outside the
scope of this book.

>While a DataFrame is physically two-dimensional, you can use it to
represent higher dimensional data in a tabular format using hierarchical
indexing, a subject we will discuss in Chapter 8 and an
ingredient in some of the more advanced data-handling features in
pandas.

---

There are many ways to construct a DataFrame, though one of the most common is
from a dict of equal-length lists or NumPy arrays:

In [74]:
data = {'state': ['Ohio', 'Ohio', 'Ohio', 'Nevada', 'Nevada', 'Nevada'],
'year': [2000, 2001, 2002, 2001, 2002, 2003],
'pop': [1.5, 1.7, 3.6, 2.4, 2.9, 3.2]}

In [75]:
frame = pd.DataFrame(data)

In [76]:
frame

,state,year,pop
0,Ohio,2000,1.5
1,Ohio,2001,1.7
2,Ohio,2002,3.6
3,Nevada,2001,2.4
4,Nevada,2002,2.9
5,Nevada,2003,3.2


If you are using the Jupyter notebook, pandas DataFrame objects will be displayed as
a more browser-friendly HTML table.

For large DataFrames, the head method selects only the first five rows:



In [77]:
frame.head()

,state,year,pop
0,Ohio,2000,1.5
1,Ohio,2001,1.7
2,Ohio,2002,3.6
3,Nevada,2001,2.4
4,Nevada,2002,2.9


If you specify a sequence of columns, the DataFrame’s columns will be arranged in
that order:

In [78]:
pd.DataFrame(data, columns=['year', 'state', 'pop'])

,year,state,pop
0,2000,Ohio,1.5
1,2001,Ohio,1.7
2,2002,Ohio,3.6
3,2001,Nevada,2.4
4,2002,Nevada,2.9
5,2003,Nevada,3.2


In [79]:
new_frame =pd.DataFrame(data, columns=['year', 'state', 'pop'])

In [80]:
new_frame

,year,state,pop
0,2000,Ohio,1.5
1,2001,Ohio,1.7
2,2002,Ohio,3.6
3,2001,Nevada,2.4
4,2002,Nevada,2.9
5,2003,Nevada,3.2


If you pass a column that isn’t contained in the dict, it will appear with missing values
in the result:

In [81]:
frame2 = pd.DataFrame(data, columns=['year', 'state', 'pop', 'debt'],index=['one', 'two', 'three', 'four','five', 'six'])

In [82]:
frame2

,year,state,pop,debt
one,2000,Ohio,1.5,NaN
two,2001,Ohio,1.7,NaN
three,2002,Ohio,3.6,NaN
four,2001,Nevada,2.4,NaN
five,2002,Nevada,2.9,NaN
six,2003,Nevada,3.2,NaN


In [83]:
frame2.columns

Index(['year', 'state', 'pop', 'debt'], dtype='object')

A column in a DataFrame can be retrieved as a Series either by dict-like notation or
by attribute:

In [84]:
frame2['state']

one        Ohio
two        Ohio
three      Ohio
four     Nevada
five     Nevada
six      Nevada
Name: state, dtype: object

In [85]:
frame2.year

one      2000
two      2001
three    2002
four     2001
five     2002
six      2003
Name: year, dtype: int64

> `frame2[column]` works for any column name, but `frame2.column`
only works when the column name is a valid Python variable
name.

Note that the returned Series have the same index as the DataFrame, and their name
attribute has been appropriately set

Rows can also be retrieved by position or name with the special loc attribute (much
more on this later):

In [86]:
frame2.loc['three']

year     2002
state    Ohio
pop       3.6
debt      NaN
Name: three, dtype: object

Columns can be modified by assignment. For example, the empty 'debt' column
could be assigned a scalar value or an array of values:

In [87]:
frame2

,year,state,pop,debt
one,2000,Ohio,1.5,NaN
two,2001,Ohio,1.7,NaN
three,2002,Ohio,3.6,NaN
four,2001,Nevada,2.4,NaN
five,2002,Nevada,2.9,NaN
six,2003,Nevada,3.2,NaN


In [88]:
frame2['debt'] = 16.5

In [89]:
c

NameError: name 'c' is not defined

In [90]:
frame2['debt'] = np.arange(6.)

In [91]:
frame2

,year,state,pop,debt
one,2000,Ohio,1.5,0.0
two,2001,Ohio,1.7,1.0
three,2002,Ohio,3.6,2.0
four,2001,Nevada,2.4,3.0
five,2002,Nevada,2.9,4.0
six,2003,Nevada,3.2,5.0


When you are assigning lists or arrays to a column, the value’s length must match the
length of the DataFrame. If you assign a Series, its labels will be realigned exactly to
the DataFrame’s index, inserting missing values in any holes:

In [92]:
val = pd.Series([-1.2, -1.5, -1.7], index=['two', 'four', 'five'])

In [93]:
frame2['debt'] = val

In [94]:
frame2

,year,state,pop,debt
one,2000,Ohio,1.5,NaN
two,2001,Ohio,1.7,-1.2
three,2002,Ohio,3.6,NaN
four,2001,Nevada,2.4,-1.5
five,2002,Nevada,2.9,-1.7
six,2003,Nevada,3.2,NaN


Assigning a column that doesn’t exist will create a new column. The del keyword will
delete columns as with a dict.
As an example of del, I first add a new column of boolean values where the state
column equals 'Ohio':

In [95]:
frame2['eastern'] = frame2.state == 'Ohio'

In [96]:
frame2

,year,state,pop,debt,eastern
one,2000,Ohio,1.5,NaN,True
two,2001,Ohio,1.7,-1.2,True
three,2002,Ohio,3.6,NaN,True
four,2001,Nevada,2.4,-1.5,False
five,2002,Nevada,2.9,-1.7,False
six,2003,Nevada,3.2,NaN,False


>New columns cannot be created with the frame2.eastern syntax.

The del method can then be used to remove this column:

In [97]:
del frame2['eastern']

In [ ]:
frame2.columns

In [ ]:
debt_vals = frame2['debt']

In [ ]:
debt_vals

In [ ]:
debt_vals.iloc[1] = 99.99

In [ ]:
debt_vals

In [ ]:
frame2

In [ ]:
# You can see 99.99 has propagated fromt he series obtained from the frame. 


>The column returned from indexing a DataFrame is a view on the
underlying data, not a copy. Thus, any in-place modifications to the
Series will be reflected in the DataFrame. The column can be
explicitly copied with the Series’s copy method.

----

In [17]:
# Nested Dictionary to Data Frame 
pop = {'Nevada': {2001: 2.4, 2002: 2.9},'Ohio': {2000: 1.5, 2001: 1.7, 2002: 3.6}}

In [18]:
pop

{'Nevada': {2001: 2.4, 2002: 2.9}, 'Ohio': {2000: 1.5, 2001: 1.7, 2002: 3.6}}

In [19]:
frame3 = pd.DataFrame(pop)

In [20]:
frame3

,Nevada,Ohio
2001,2.4,1.7
2002,2.9,3.6
2000,NaN,1.5


If the nested dict is passed to the DataFrame, pandas will interpret the outer dict keys
as the columns and the inner keys as the row indices:

----

You can transpose the DataFrame (swap rows and columns) with similar syntax to a
NumPy array:

In [23]:
frame3.T

,2001,2002,2000
Nevada,2.4,2.9,NaN
Ohio,1.7,3.6,1.5


In [24]:
frame3['Ohio'][:-1]

2001    1.7
2002    3.6
Name: Ohio, dtype: float64

------
Dicts of Series are treated in much the same way:

In [25]:
pdata = {'Ohio': [1.6, 3.7, 2.2], 'Nevada':[6.1, 7.3, 2.2]}

In [26]:
pd.DataFrame(pdata)

,Ohio,Nevada
0,1.6,6.1
1,3.7,7.3
2,2.2,2.2


In [27]:
frame3['Ohio'][:-1]

2001    1.7
2002    3.6
Name: Ohio, dtype: float64

In [28]:
frame3['Nevada'][:2]

2001    2.4
2002    2.9
Name: Nevada, dtype: float64

In [32]:
pdata1 = {'Ohio': frame3['Ohio'][:-1],'Nevada': frame3['Nevada'][:2]}

In [33]:
pdata1

{'Ohio': 2001    1.7
 2002    3.6
 Name: Ohio, dtype: float64,
 'Nevada': 2001    2.4
 2002    2.9
 Name: Nevada, dtype: float64}

In [34]:
pd.DataFrame(pdata1)

,Ohio,Nevada
2001,1.7,2.4
2002,3.6,2.9


If you see the difference between `pdata` and `pdata1` , `pdata` is a dict of simple array of numbers where as `pdata1` is a dictionary of Panda Sequences. And Since panda sequence has index and here index is year numbers, that become index fo data frame.  This is my `pdata` has only 0,1,2 indices where as `pdata1` has 2002,2002 etc... 


**Possible data inputs to DataFrame constructor**

In [3]:
columns = ["Type", "Notes"]

rows = [
["2D ndarray","Matrix of data with optional row and column labels"],
["dict of arrays / lists / tuples","Each sequence becomes a column; all sequences must be same length"],
["NumPy structured / record array","Treated like dict of arrays"],
["dict of Series","Each value becomes a column; indexes are unioned to form row index"],
["dict of dicts","Each inner dict becomes a column; keys are unioned as row index"],
["List of dicts or Series","Each item becomes a row; union of keys/indexes becomes column labels"],
["List of lists or tuples","Treated like 2D ndarray"],
["Another DataFrame","Indexes are reused unless new ones provided"],
["NumPy MaskedArray","Like 2D ndarray but masked values become NA"]
]

render_book_table(
    "Table 5-1. Possible Data Inputs to DataFrame Constructor",
    columns,
    rows
)

Type,Notes
2D ndarray,Matrix of data with optional row and column labels
dict of arrays / lists / tuples,Each sequence becomes a column; all sequences must be same length
NumPy structured / record array,Treated like dict of arrays
dict of Series,Each value becomes a column; indexes are unioned to form row index
dict of dicts,Each inner dict becomes a column; keys are unioned as row index
List of dicts or Series,Each item becomes a row; union of keys/indexes becomes column labels
List of lists or tuples,Treated like 2D ndarray
Another DataFrame,Indexes are reused unless new ones provided
NumPy MaskedArray,Like 2D ndarray but masked values become NA


----

If a DataFrame’s index and columns have their name attributes set, these will also be
displayed:

In [59]:
frame3.index.name = 'year'; frame3.columns.name = 'state'

In [61]:
frame3

state,Nevada,Ohio
year,,
2001,2.4,1.7
2002,2.9,3.6
2000,NaN,1.5


-----
As with Series, the values attribute returns the data contained in the DataFrame as a
two-dimensional ndarray:

In [62]:
frame3.values

array([[2.4, 1.7],
       [2.9, 3.6],
       [nan, 1.5]])

If the DataFrame’s columns are different dtypes, the dtype of the values array will be
chosen to accommodate all of the columns:

In [98]:
frame2.values

array([[2000, 'Ohio', 1.5, nan],
       [2001, 'Ohio', 1.7, -1.2],
       [2002, 'Ohio', 3.6, nan],
       [2001, 'Nevada', 2.4, -1.5],
       [2002, 'Nevada', 2.9, -1.7],
       [2003, 'Nevada', 3.2, nan]], dtype=object)

### Index Objects

Pandas’s Index objects are responsible for holding the axis labels and other metadata
(like the axis name or names). Any array or other sequence of labels you use when
constructing a Series or DataFrame is internally converted to an Index:

In [100]:
obj = pd.Series(range(3), index=['a', 'b', 'c'])

In [102]:
index_joe = obj.index

In [103]:
index_joe

Index(['a', 'b', 'c'], dtype='object')

In [105]:
index_joe[1:]

Index(['b', 'c'], dtype='object')

Index objects are immutable and thus can’t be modified by the user:

In [106]:
index_joe[1] = 'jpj'
#TypeError: Index does not support mutable operations

TypeError: Index does not support mutable operations

Immutability makes it safer to share Index objects among data structures:

In [108]:
labels = pd.Index(np.arange(3)) # Note you can create an index not associated with a data frame using pd.Index

In [109]:
labels

Index([0, 1, 2], dtype='int64')

In [110]:
obj2 = pd.Series([1.5, -2.5, 0], index=labels)

In [111]:
obj2.index

Index([0, 1, 2], dtype='int64')

In [113]:
obj2.index.name='Joyan'

In [114]:
obj2.index

Index([0, 1, 2], dtype='int64', name='Joyan')

In [115]:
obj2

Joyan
0    1.5
1   -2.5
2    0.0
dtype: float64

AttributeError: 'Series' object has no attribute 'column'

In [121]:
obj3 = pd.DataFrame([1.5, -2.5, 0], index=labels, columns = ['values'])

In [122]:
obj3

,values
Joyan,
0,1.5
1,-2.5
2,0.0


In [123]:
obj3.columns

Index(['values'], dtype='object')

In [125]:
obj3.columns.name = 'Julia'

In [126]:
obj3

Julia,values
Joyan,
0,1.5
1,-2.5
2,0.0


In [127]:
obj2.index is labels

True

In [128]:
obj3.index is labels

True

In [129]:
#In addition to being array-like, an Index also behaves like a fixed-size set:

In [130]:
frame3

state,Nevada,Ohio
year,,
2001,2.4,1.7
2002,2.9,3.6
2000,NaN,1.5


In [131]:
frame3.columns

Index(['Nevada', 'Ohio'], dtype='object', name='state')

In [132]:
'Ohio' in frame3.columns

True

In [134]:
frame3.index

Index([2001, 2002, 2000], dtype='int64', name='year')

In [135]:
'2001' in frame3.index

False

In [136]:
frame3.index

Index([2001, 2002, 2000], dtype='int64', name='year')

In [138]:
2001 in frame3.index # the Data type of Index in this case was 

True

Unlike Python sets, a pandas Index can contain duplicate labels:

In [139]:
dup_labels = pd.Index(['foo', 'foo', 'bar', 'bar'])

In [140]:
dup_labels

Index(['foo', 'foo', 'bar', 'bar'], dtype='object')

In [141]:
index_like_set = {'foo', 'foo', 'bar', 'bar'}

In [142]:
index_like_set

{'bar', 'foo'}

In [143]:
data = {'state': ['Kerala', 'Karnataka', 'Tamil Nadu', 'Goas'],
'year': [2000, 2001, 2002, 2003, ],
'pop': [1.5, 1.7, 3.6, 2.4]}

In [144]:
data

{'state': ['Kerala', 'Karnataka', 'Tamil Nadu', 'Goas'],
 'year': [2000, 2001, 2002, 2003],
 'pop': [1.5, 1.7, 3.6, 2.4]}

In [146]:
frame5 = pd.DataFrame(data)

In [150]:
frame5.index=dup_labels

In [151]:
frame5

,state,year,pop
foo,Kerala,2000,1.5
foo,Karnataka,2001,1.7
bar,Tamil Nadu,2002,3.6
bar,Goas,2003,2.4


In [162]:
frame5.loc['foo']

,state,year,pop
foo,Kerala,2000,1.5
foo,Karnataka,2001,1.7


In [164]:
frame5.index.unique()

Index(['foo', 'bar'], dtype='object')

Each Index has a number of methods and properties for set logic, which answer other
common questions about the data it contains. Some useful ones are summarized Below

In [165]:
import pandas as pd

data = [
    ("append", "Concatenate with additional Index objects, producing a new Index"),
    ("difference", "Compute set difference as an Index"),
    ("intersection", "Compute set intersection"),
    ("union", "Compute set union"),
    ("isin", "Compute boolean array indicating whether each value is contained in a collection"),
    ("delete", "Compute new Index with element at index i deleted"),
    ("drop", "Compute new Index by deleting passed values"),
    ("insert", "Compute new Index by inserting element at index i"),
    ("is_monotonic", "Returns True if each element is greater than or equal to the previous one"),
    ("is_unique", "Returns True if the Index has no duplicate values"),
    ("unique", "Compute the array of unique values in the Index"),
]

df = pd.DataFrame(data, columns=["Method", "Description"])
df.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white")]}]
)

,Method,Description
0,append,"Concatenate with additional Index objects, producing a new Index"
1,difference,Compute set difference as an Index
2,intersection,Compute set intersection
3,union,Compute set union
4,isin,Compute boolean array indicating whether each value is contained in a collection
5,delete,Compute new Index with element at index i deleted
6,drop,Compute new Index by deleting passed values
7,insert,Compute new Index by inserting element at index i
8,is_monotonic,Returns True if each element is greater than or equal to the previous one
9,is_unique,Returns True if the Index has no duplicate values


## 5.2 Essential Functionality

This section will walk you through the fundamental mechanics of interacting with the
data contained in a Series or DataFrame. In the chapters to come, we will delve more
deeply into data analysis and manipulation topics using pandas. This book is not
intended to serve as exhaustive documentation for the pandas library; instead, we’ll
focus on the most important features, leaving the less common (i.e., more esoteric)
things for you to explore on your own.

### Reindexing

An important method on pandas objects is reindex, which means to create a new
object with the data conformed to a new index.

In [166]:
obj = pd.Series([4.5, 7.2, -5.3, 3.6], index=['d', 'b', 'a', 'c'])

In [168]:
obj

d    4.5
b    7.2
a   -5.3
c    3.6
dtype: float64

Calling reindex on this Series rearranges the data according to the new index, introducing
missing values if any index values were not already present:

In [169]:
obj.reindex(['a', 'b', 'c', 'd', 'e'])

a   -5.3
b    7.2
c    3.6
d    4.5
e    NaN
dtype: float64

In [170]:
obj  # you can see that it didnt replace.

d    4.5
b    7.2
a   -5.3
c    3.6
dtype: float64

reindex() does NOT have an inplace parameter in pandas.

Why?

reindex() is a structural transformation:

It can change the shape

It can introduce new labels

It can insert NaN values

It can change row/column alignment

Because of this, pandas treats it as a pure function:
it returns a new object instead of mutating the original one.

This aligns with pandas’ design philosophy that most label-altering operations return new objects.

In [171]:
obj2 = obj.reindex(['a', 'b', 'c', 'd', 'e'])

In [172]:
obj2

a   -5.3
b    7.2
c    3.6
d    4.5
e    NaN
dtype: float64

----

For ordered data like time series, it may be desirable to do some interpolation or filling
of values when reindexing. The method option allows us to do this, using a
method such as `ffill`, which forward-fills the values:

In [173]:
obj3 = pd.Series(['blue', 'purple', 'yellow'], index=[0, 2, 4])

In [174]:
obj3

0      blue
2    purple
4    yellow
dtype: object

In [175]:
obj3.reindex(range(6), method='ffill')

0      blue
1      blue
2    purple
3    purple
4    yellow
5    yellow
dtype: object

With DataFrame, reindex can alter either the (row) index, columns, or both. When
passed only a sequence, it reindexes the rows in the result:

In [176]:
frame = pd.DataFrame(np.arange(9).reshape((3, 3)), index=['a', 'c', 'd'],columns=['Ohio', 'Texas', 'California'])

In [177]:
frame

,Ohio,Texas,California
a,0,1,2
c,3,4,5
d,6,7,8


In [178]:
frame2 = frame.reindex(['a', 'b', 'c', 'd'])


In [179]:
frame2

,Ohio,Texas,California
a,0.0,1.0,2.0
b,NaN,NaN,NaN
c,3.0,4.0,5.0
d,6.0,7.0,8.0


The columns can be reindexed with the columns keyword:

In [180]:
states = ['Texas', 'Utah', 'California']
frame.reindex(columns=states)

,Texas,Utah,California
a,1,NaN,2
c,4,NaN,5
d,7,NaN,8


As we’ll explore in more detail, you can reindex more succinctly by label-indexing
with loc, and many users prefer to use it exclusively:

----

See Table  for more about the arguments to reindex.

In [185]:
import pandas as pd

data = [
    ("index", "New sequence to use as index. Can be Index instance or any sequence-like structure. An Index is used as-is without copying."),
    ("method", "Interpolation method: 'ffill' fills forward, 'bfill' fills backward."),
    ("fill_value", "Value to substitute when introducing missing data during reindexing."),
    ("limit", "Maximum number of consecutive elements to fill when forward/backward filling."),
    ("tolerance", "Maximum numeric distance allowed for inexact matches when filling."),
    ("level", "Match simple Index on a specific level of a MultiIndex."),
    ("copy", "If True, always copy data even if index is unchanged; if False, avoid copy when possible.")
]

df = pd.DataFrame(data, columns=["Argument", "Description"])

with pd.option_context('display.max_colwidth', None):
    display(
        df.style
          .set_properties(**{
              "text-align": "left",
              "white-space": "normal"
          })
          .set_table_styles([
              {
                  "selector": "th",
                  "props": [
                      ("text-align", "left"),
                      ("background-color", "#2c3e50"),
                      ("color", "white"),
                      ("font-weight", "bold")
                  ]
              }
          ])
    )

,Argument,Description
0,index,New sequence to use as index. Can be Index instance or any sequence-like structure. An Index is used as-is without copying.
1,method,"Interpolation method: 'ffill' fills forward, 'bfill' fills backward."
2,fill_value,Value to substitute when introducing missing data during reindexing.
3,limit,Maximum number of consecutive elements to fill when forward/backward filling.
4,tolerance,Maximum numeric distance allowed for inexact matches when filling.
5,level,Match simple Index on a specific level of a MultiIndex.
6,copy,"If True, always copy data even if index is unchanged; if False, avoid copy when possible."


### Dropping Entries from an Axis

Dropping one or more entries from an axis is easy if you already have an index array
or list without those entries. As that can require a bit of munging and set logic, the
138 | Chapter 5: Getting Started with pandas
drop method will return a new object with the indicated value or values deleted from
an axis:

In [186]:
obj = pd.Series(np.arange(5.), index=['a', 'b', 'c', 'd', 'e'])

In [187]:
obj

a    0.0
b    1.0
c    2.0
d    3.0
e    4.0
dtype: float64

In [188]:
new_obj = obj.drop('c')

In [189]:
new_obj

a    0.0
b    1.0
d    3.0
e    4.0
dtype: float64

In [190]:
obj # It doesnt remove from original one.

a    0.0
b    1.0
c    2.0
d    3.0
e    4.0
dtype: float64

In [191]:
obj.drop(['d', 'c'])

a    0.0
b    1.0
e    4.0
dtype: float64

In [192]:
data = pd.DataFrame(np.arange(16).reshape((4, 4)), index=['Ohio', 'Colorado', 'Utah', 'New York'], columns=['one', 'two', 'three', 'four'])

In [193]:
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


Calling drop with a sequence of labels will drop values from the row labels (axis 0):

In [194]:
data.drop(['Colorado', 'Ohio'])

,one,two,three,four
Utah,8,9,10,11
New York,12,13,14,15


You can drop values from the columns by passing `axis=1` or `axis='columns'`

In [196]:
data.drop('two', axis=1)

,one,three,four
Ohio,0,2,3
Colorado,4,6,7
Utah,8,10,11
New York,12,14,15


In [197]:
data.drop(['two', 'four'], axis='columns')

,one,three
Ohio,0,2
Colorado,4,6
Utah,8,10
New York,12,14


Many functions, like drop, which modify the size or shape of a Series or DataFrame,
can manipulate an object in-place without returning a new object:


In [205]:
data = pd.DataFrame(np.arange(16).reshape((4, 4)), index=['Ohio', 'Colorado', 'Utah', 'New York'], columns=['one', 'two', 'three', 'four'])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [206]:
data.drop(['Ohio'])


,one,two,three,four
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [207]:
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [208]:
data.drop(['Ohio'], inplace=True)

In [209]:
data

,one,two,three,four
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


> **Be careful with the inplace, as it destroys any data that is dropped.***

---

### Indexing, Selection, and Filtering

Series indexing (`obj[...]`) works analogously to NumPy array indexing, except you
can use the Series’s index values instead of only integers. Here are some examples of
this:

In [5]:
obj = pd.Series(np.arange(4.), index=['a', 'b', 'c', 'd'])

In [6]:
obj

a    0.0
b    1.0
c    2.0
d    3.0
dtype: float64

In [7]:
obj['b']

np.float64(1.0)

In [8]:
obj[1]

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_4064\2469632899.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  obj[1]


np.float64(1.0)

In [9]:
obj[2:4]

c    2.0
d    3.0
dtype: float64

In [10]:
obj[['b', 'a', 'd']]

b    1.0
a    0.0
d    3.0
dtype: float64

In [11]:
obj[[1, 3]]

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_4064\2982346117.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  obj[[1, 3]]


b    1.0
d    3.0
dtype: float64

In [12]:
obj[[1, 3]]

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_4064\2982346117.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  obj[[1, 3]]


b    1.0
d    3.0
dtype: float64

In [13]:
obj < 2

a     True
b     True
c    False
d    False
dtype: bool

In [14]:
obj[obj < 2]

a    0.0
b    1.0
dtype: float64

Slicing with labels behaves differently than normal Python slicing in that the endpoint
is inclusive:

In [15]:
obj['b':'c']

b    1.0
c    2.0
dtype: float64

Setting using these methods modifies the corresponding section of the Series:

In [16]:
obj['b':'c'] = 5

In [18]:
obj

a    0.0
b    5.0
c    5.0
d    3.0
dtype: float64

----

Indexing into a DataFrame is for retrieving one or more columns either with a single
value or sequence:

In [19]:
data = pd.DataFrame(np.arange(16).reshape((4, 4)),index=['Ohio', 'Colorado', 'Utah', 'New York'],columns=['one', 'two', 'three', 'four'])

In [20]:
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [21]:
data['two']

Ohio         1
Colorado     5
Utah         9
New York    13
Name: two, dtype: int64

In [22]:
data['Ohio']

KeyError: 'Ohio'

In [23]:
data[['three', 'one']]

,three,one
Ohio,2,0
Colorado,6,4
Utah,10,8
New York,14,12


Indexing like this has a few special cases. First, slicing or selecting data with a boolean
array:

In [25]:
data[:2]  # This selects rows 0 and 1

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7


In [26]:
data[data['three'] > 5] # Picks all columns and rows where colum3 value is > 5. 

,one,two,three,four
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


The row selection syntax data[:2] is provided as a convenience. Passing a single element
or a list to the [] operator selects columns.

---

Another use case is in indexing with a boolean DataFrame, such as one produced by a
scalar comparison:

In [28]:
data < 5

,one,two,three,four
Ohio,True,True,True,True
Colorado,True,False,False,False
Utah,False,False,False,False
New York,False,False,False,False


In [29]:
data[data < 5] = 0

In [30]:
data

,one,two,three,four
Ohio,0,0,0,0
Colorado,0,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


This makes DataFrame syntactically more like a two-dimensional NumPy array in
this particular case.

#### Selection with loc and iloc

For DataFrame label-indexing on the rows, I introduce the special indexing operators
loc and iloc. They enable you to select a subset of the rows and columns from a
DataFrame with NumPy-like notation using either axis labels (loc) or integers
(iloc).

As a preliminary example, let’s select a single row and multiple columns by label:

In [31]:
data.loc['Colorado', ['two', 'three']]

two      5
three    6
Name: Colorado, dtype: int64

In [33]:
# We’ll then perform some similar selections with integers using iloc:
data.iloc[2, [3, 0, 1]] # 2nd row columsn 3, 0 and 1

four    11
one      8
two      9
Name: Utah, dtype: int64

In [34]:
data.iloc[2] # Only rwo selected

one       8
two       9
three    10
four     11
Name: Utah, dtype: int64

In [35]:
data.iloc[[1, 2], [3, 0, 1]] ## 2 Rows, 3 columns

,four,one,two
Colorado,7,0,5
Utah,11,8,9


In [36]:
data.loc['Utah', 'two']

np.int64(9)

In [37]:
data.loc[['Utah', 'Colorado'], 'two']

Utah        9
Colorado    5
Name: two, dtype: int64

In [38]:
data.loc[:'Utah', 'two']  ## Till Utah and then column 2

Ohio        0
Colorado    5
Utah        9
Name: two, dtype: int64

In [39]:
data.iloc[:, :3][data.three > 5]   # akk rows, then first 3 columns, but only for records where column 3 is >5

,one,two,three
Colorado,0,5,6
Utah,8,9,10
New York,12,13,14


So there are many ways to select and rearrange the data contained in a pandas object.
For DataFrame, Table belowprovides a short summary of many of them. As you’ll see
later, there are a number of additional options for working with hierarchical indexes.

In [40]:
data = [
    ("df[val]", "Select single column or sequence of columns from the DataFrame; special cases: boolean array (filter rows), slice (slice rows), or boolean DataFrame (set values based on some criterion)."),
    ("df.loc[val]", "Select single row or subset of rows from the DataFrame by label."),
    ("df.loc[:, val]", "Select single column or subset of columns by label."),
    ("df.loc[val1, val2]", "Select both rows and columns by label."),
    ("df.iloc[where]", "Select single row or subset of rows from the DataFrame by integer position."),
    ("df.iloc[:, where]", "Select single column or subset of columns by integer position."),
    ("df.iloc[where_i, where_j]", "Select both rows and columns by integer position."),
    ("df.at[label_i, label_j]", "Select a single scalar value by row and column label."),
    ("df.iat[i, j]", "Select a single scalar value by row and column position (integers)."),
    ("reindex method", "Select either rows or columns by labels."),
    ("get_value, set_value methods", "Select or set a single value by row and column label (legacy methods).")
]

df = pd.DataFrame(data, columns=["Type", "Notes"])

with pd.option_context('display.max_colwidth', None):
    display(
        df.style
          .set_properties(**{
              "text-align": "left",
              "white-space": "normal"
          })
          .set_table_styles([
              {
                  "selector": "th",
                  "props": [
                      ("text-align", "left"),
                      ("background-color", "#2c3e50"),
                      ("color", "white"),
                      ("font-weight", "bold")
                  ]
              }
          ])
    )

,Type,Notes
0,df[val],"Select single column or sequence of columns from the DataFrame; special cases: boolean array (filter rows), slice (slice rows), or boolean DataFrame (set values based on some criterion)."
1,df.loc[val],Select single row or subset of rows from the DataFrame by label.
2,"df.loc[:, val]",Select single column or subset of columns by label.
3,"df.loc[val1, val2]",Select both rows and columns by label.
4,df.iloc[where],Select single row or subset of rows from the DataFrame by integer position.
5,"df.iloc[:, where]",Select single column or subset of columns by integer position.
6,"df.iloc[where_i, where_j]",Select both rows and columns by integer position.
7,"df.at[label_i, label_j]",Select a single scalar value by row and column label.
8,"df.iat[i, j]",Select a single scalar value by row and column position (integers).
9,reindex method,Select either rows or columns by labels.


#### Integer Indexes

In [48]:
ser = pd.Series(np.arange(3.))

In [49]:
ser[1]

np.float64(1.0)

In [50]:
ser[-1]

KeyError: -1

you might not expect the following code
to generate an error: In this case, pandas could “fall back” on integer indexing, but it’s difficult to do this in
general without introducing subtle bugs. Here we have an index containing 0, 1, 2,
but inferring what the user wants (label-based indexing or position-based) is difficult:

In [51]:
ser

0    0.0
1    1.0
2    2.0
dtype: float64

On the other hand, with a non-integer index, there is no potential for ambiguity:

In [53]:
ser2 = pd.Series(np.arange(3.), index=['a', 'b', 'c'])

In [54]:
ser2[-1]

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_4064\811950851.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ser2[-1]


np.float64(2.0)

To keep things consistent, if you have an axis index containing integers, data selection
will always be label-oriented. For more precise handling, use loc (for labels) or iloc
(for integers):

In [55]:
ser[:1]

0    0.0
dtype: float64

In [56]:
ser.loc[:1]

0    0.0
1    1.0
dtype: float64

In [57]:
ser.iloc[:1]

0    0.0
dtype: float64

### Arithmetic and Data Alignment

An important pandas feature for some applications is the behavior of arithmetic
between objects with different indexes. When you are adding together objects, if any
index pairs are not the same, the respective index in the result will be the union of the
index pairs. For users with database experience, this is similar to an automatic outer
join on the index labels. Let’s look at an example:

In [58]:
s1 = pd.Series([7.3, -2.5, 3.4, 1.5], index=['a', 'c', 'd', 'e'])

In [59]:
s2 = pd.Series([-2.1, 3.6, -1.5, 4, 3.1], index=['a', 'c', 'e', 'f', 'g'])

In [60]:
s1

a    7.3
c   -2.5
d    3.4
e    1.5
dtype: float64

In [61]:
s2

a   -2.1
c    3.6
e   -1.5
f    4.0
g    3.1
dtype: float64

In [62]:
s1+s2

a    5.2
c    1.1
d    NaN
e    0.0
f    NaN
g    NaN
dtype: float64

That is : The internal data alignment introduces missing values in the label locations that don’t
overlap. Missing values will then propagate in further arithmetic computations.


------
In the case of DataFrame, alignment is performed on both the rows and the columns:

In [64]:
df1 = pd.DataFrame(np.arange(9.).reshape((3, 3)), columns=list('bcd'),index=['Ohio', 'Texas', 'Colorado'])

In [65]:
df2 = pd.DataFrame(np.arange(12.).reshape((4, 3)), columns=list('bde'),index=['Utah', 'Ohio', 'Texas', 'Oregon'])

In [66]:
df1

,b,c,d
Ohio,0.0,1.0,2.0
Texas,3.0,4.0,5.0
Colorado,6.0,7.0,8.0


In [67]:
df2

,b,d,e
Utah,0.0,1.0,2.0
Ohio,3.0,4.0,5.0
Texas,6.0,7.0,8.0
Oregon,9.0,10.0,11.0


In [68]:
df1+df2

,b,c,d,e
Colorado,NaN,NaN,NaN,NaN
Ohio,3.0,NaN,6.0,NaN
Oregon,NaN,NaN,NaN,NaN
Texas,9.0,NaN,12.0,NaN
Utah,NaN,NaN,NaN,NaN


Since the 'c' and 'e' columns are not found in both DataFrame objects, they appear
as all missing in the result. The same holds for the rows whose labels are not common
to both objects.

If you add DataFrame objects with no column or row labels in common, the result
will contain all nulls:

#### Arithmetic methods with fill values

In arithmetic operations between differently indexed objects, you might want to fill
with a special value, like 0, when an axis label is found in one object but not the other:

In [69]:
df1 = pd.DataFrame(np.arange(12.).reshape((3, 4)),columns=list('abcd'))

In [70]:
df2 = pd.DataFrame(np.arange(20.).reshape((4, 5)),columns=list('abcde'))


In [71]:
df2.loc[1, 'b'] = np.nan

In [72]:
df1

,a,b,c,d
0,0.0,1.0,2.0,3.0
1,4.0,5.0,6.0,7.0
2,8.0,9.0,10.0,11.0


In [73]:
df2

,a,b,c,d,e
0,0.0,1.0,2.0,3.0,4.0
1,5.0,NaN,7.0,8.0,9.0
2,10.0,11.0,12.0,13.0,14.0
3,15.0,16.0,17.0,18.0,19.0


In [74]:
df1 + df2

,a,b,c,d,e
0,0.0,2.0,4.0,6.0,NaN
1,9.0,NaN,13.0,15.0,NaN
2,18.0,20.0,22.0,24.0,NaN
3,NaN,NaN,NaN,NaN,NaN


Using the add method on df1, I pass df2 and an argument to fill_value:

In [75]:
df1.add(df2, fill_value=0)

,a,b,c,d,e
0,0.0,2.0,4.0,6.0,4.0
1,9.0,5.0,13.0,15.0,9.0
2,18.0,20.0,22.0,24.0,14.0
3,15.0,16.0,17.0,18.0,19.0


See Table below  for a listing of Series and DataFrame methods for arithmetic. Each of
them has a counterpart, starting with the letter r, that has arguments flipped. So these
two statements are equivalent:

In [77]:
import pandas as pd

data = [
    ("add, radd", "Methods for addition (+)."),
    ("sub, rsub", "Methods for subtraction (-)."),
    ("div, rdiv", "Methods for division (/)."),
    ("floordiv, rfloordiv", "Methods for floor division (//)."),
    ("mul, rmul", "Methods for multiplication (*)."),
    ("pow, rpow", "Methods for exponentiation (**).")
]

df = pd.DataFrame(data, columns=["Method", "Description"])

with pd.option_context('display.max_colwidth', None):
    display(
        df.style
          .set_properties(**{
              "text-align": "left",
              "white-space": "normal"
          })
          .set_table_styles([
              {
                  "selector": "th",
                  "props": [
                      ("text-align", "left"),
                      ("background-color", "#2c3e50"),
                      ("color", "white"),
                      ("font-weight", "bold")
                  ]
              }
          ])
    )

,Method,Description
0,"add, radd",Methods for addition (+).
1,"sub, rsub",Methods for subtraction (-).
2,"div, rdiv",Methods for division (/).
3,"floordiv, rfloordiv",Methods for floor division (//).
4,"mul, rmul",Methods for multiplication (*).
5,"pow, rpow",Methods for exponentiation (**).


See how "r" works.

In [78]:
1 / df1

,a,b,c,d
0,inf,1.000000,0.500000,0.333333
1,0.250,0.200000,0.166667,0.142857
2,0.125,0.111111,0.100000,0.090909


In [79]:
df1.rdiv(1)

,a,b,c,d
0,inf,1.000000,0.500000,0.333333
1,0.250,0.200000,0.166667,0.142857
2,0.125,0.111111,0.100000,0.090909


#### Operations between DataFrame and Series

As with NumPy arrays of different dimensions, arithmetic between DataFrame and
Series is also defined. First, as a motivating example, consider the difference between
a two-dimensional array and one of its rows:

In [5]:
arr = np.arange(12.).reshape((3, 4))

In [6]:
arr

array([[ 0.,  1.,  2.,  3.],
       [ 4.,  5.,  6.,  7.],
       [ 8.,  9., 10., 11.]])

In [7]:
arr[0]

array([0., 1., 2., 3.])

In [8]:
arr - arr[0]

array([[0., 0., 0., 0.],
       [4., 4., 4., 4.],
       [8., 8., 8., 8.]])

When we subtract arr[0] from arr, the subtraction is performed once for each row.
This is referred to as broadcasting and is explained in more detail as it relates to general
NumPy arrays in Appendix A. 

--------------------
Operations between a DataFrame and a Series are
similar:

In [11]:
frame = pd.DataFrame(np.arange(12.).reshape((4, 3)), columns=list('bde'), index=['Utah', 'Ohio', 'Texas', 'Oregon'])

In [12]:
series = frame.iloc[0]

In [13]:
frame

,b,d,e
Utah,0.0,1.0,2.0
Ohio,3.0,4.0,5.0
Texas,6.0,7.0,8.0
Oregon,9.0,10.0,11.0


In [14]:
series

b    0.0
d    1.0
e    2.0
Name: Utah, dtype: float64

In [15]:
frame-series

,b,d,e
Utah,0.0,0.0,0.0
Ohio,3.0,3.0,3.0
Texas,6.0,6.0,6.0
Oregon,9.0,9.0,9.0


By default, arithmetic between DataFrame and Series matches the **index of the Series
on the DataFrame’s columns**, broadcasting down the rows:

If an index value is not found in either the DataFrame’s columns or the Series’s index,
the objects will be reindexed to form the union:

In [17]:
series2 = pd.Series(range(3), index=['b', 'e', 'f'])

In [18]:
series2

b    0
e    1
f    2
dtype: int64

In [19]:
frame-series2

,b,d,e,f
Utah,0.0,NaN,1.0,NaN
Ohio,3.0,NaN,4.0,NaN
Texas,6.0,NaN,7.0,NaN
Oregon,9.0,NaN,10.0,NaN


If you want to instead broadcast over the columns, matching on the rows, you have to
use one of the arithmetic methods. For example:

In [20]:
series3 = frame['d']

In [21]:
frame

,b,d,e
Utah,0.0,1.0,2.0
Ohio,3.0,4.0,5.0
Texas,6.0,7.0,8.0
Oregon,9.0,10.0,11.0


In [22]:
series3

Utah       1.0
Ohio       4.0
Texas      7.0
Oregon    10.0
Name: d, dtype: float64

In [23]:
frame.sub(series3, axis='index') # 

,b,d,e
Utah,-1.0,0.0,1.0
Ohio,-1.0,0.0,1.0
Texas,-1.0,0.0,1.0
Oregon,-1.0,0.0,1.0


The axis number that you pass is the axis to match on. In this case we mean to match
on the DataFrame’s row index (`axis='index'` or `axis=0`) and broadcast across.





----

### Function Application and Mapping

NumPy ufuncs (element-wise array methods) also work with pandas objects:

In [28]:
frame = pd.DataFrame(np.random.randn(4, 3), columns=list('bde'), index=['Utah', 'Ohio', 'Texas', 'Oregon'])

`np.random.randn(4, 3)` is a ufunc.



In [31]:
frame

,b,d,e
Utah,-0.233288,1.376857,1.021765
Ohio,0.504014,1.865312,0.208364
Texas,0.321466,-0.469851,0.749130
Oregon,-0.295629,0.213720,0.029662


In [32]:
np.abs(frame)

,b,d,e
Utah,0.233288,1.376857,1.021765
Ohio,0.504014,1.865312,0.208364
Texas,0.321466,0.469851,0.749130
Oregon,0.295629,0.213720,0.029662


In [33]:
f = lambda x: x.max() - x.min()

In [34]:
frame.apply(f)

b    0.799644
d    2.335163
e    0.992103
dtype: float64

Here the function `f`, which computes the difference between the maximum and minimum
of a Series, is invoked once on each column in `frame`. The result is a Series having
the columns of `frame` as its index.

If you pass `axis='columns'` to apply, the function will be invoked once per row
instead:

In [37]:
frame

,b,d,e
Utah,-0.233288,1.376857,1.021765
Ohio,0.504014,1.865312,0.208364
Texas,0.321466,-0.469851,0.749130
Oregon,-0.295629,0.213720,0.029662


In [38]:
frame.apply(f, axis='columns')

Utah      1.610145
Ohio      1.656948
Texas     1.218981
Oregon    0.509349
dtype: float64

Many of the most common array statistics (like sum and mean) are DataFrame methods,
so using apply is not necessary.

The function passed to `apply` need not return a scalar value; it can also return a Series
with multiple values:

In [40]:
def f(x):
    return pd.Series([x.min(), x.max()], index=['min', 'max'])

In [41]:
frame.apply(f)

,b,d,e
min,-0.295629,-0.469851,0.029662
max,0.504014,1.865312,1.021765


Element-wise Python functions can be used, too. Suppose you wanted to compute a
formatted string from each floating-point value in frame. You can do this with `apply
map` or `map`:

In [42]:
format = lambda x: '%.2f' % x

In [44]:
frame

,b,d,e
Utah,-0.233288,1.376857,1.021765
Ohio,0.504014,1.865312,0.208364
Texas,0.321466,-0.469851,0.749130
Oregon,-0.295629,0.213720,0.029662


In [45]:
frame.map(format)

,b,d,e
Utah,-0.23,1.38,1.02
Ohio,0.50,1.87,0.21
Texas,0.32,-0.47,0.75
Oregon,-0.30,0.21,0.03


In [48]:
c.applymap(format)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_40336\304758519.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  frame.applymap(format)


,b,d,e
Utah,-0.23,1.38,1.02
Ohio,0.50,1.87,0.21
Texas,0.32,-0.47,0.75
Oregon,-0.30,0.21,0.03


In [50]:
frame['d'].map(format)

Utah       1.38
Ohio       1.87
Texas     -0.47
Oregon     0.21
Name: d, dtype: object

### Sorting and Ranking

Sorting a dataset by some criterion is another important built-in operation. To sort
lexicographically by row or column index, use the `sort_index` method, which returns
a new, sorted object:

In [52]:
obj = pd.Series(range(4), index=['d', 'a', 'b', 'c'])

In [53]:
obj

d    0
a    1
b    2
c    3
dtype: int64

In [54]:
obj.sort_index()

a    1
b    2
c    3
d    0
dtype: int64

---

With a DataFrame, you can sort by index on either axis:

In [55]:
frame = pd.DataFrame(np.arange(8).reshape((2, 4)), index=['three', 'one'],columns=['d', 'a', 'b', 'c'])

In [56]:
frame

,d,a,b,c
three,0,1,2,3
one,4,5,6,7


In [57]:
frame.sort_index()

,d,a,b,c
one,4,5,6,7
three,0,1,2,3


In [58]:
frame.sort_index(axis=1)

,a,b,c,d
three,1,2,3,0
one,5,6,7,4


The data is sorted in ascending order by default, but can be sorted in descending
order, too:

In [59]:
frame.sort_index(axis=1, ascending=False)

,d,c,b,a
three,0,3,2,1
one,4,7,6,5


To sort a Series by its values, use its `sort_values` method:

In [61]:
obj = pd.Series([4, 7, -3, 2])

In [62]:
obj.sort_values()

2   -3
3    2
0    4
1    7
dtype: int64

Any missing values are sorted to the end of the Series by default:

In [64]:
obj = pd.Series([4, np.nan, 7, np.nan, -3, 2])

In [65]:
obj.sort_values()

4   -3.0
5    2.0
0    4.0
2    7.0
1    NaN
3    NaN
dtype: float64

----

When sorting a DataFrame, you can use the data in one or more columns as the sort
keys. To do so, pass one or more column names to the by option of sort_values:

In [66]:
frame = pd.DataFrame({'b': [4, 7, -3, 2], 'a': [0, 1, 0, 1]})

In [67]:
frame

,b,a
0,4,0
1,7,1
2,-3,0
3,2,1


In [68]:
frame.sort_values(by='b')

,b,a
2,-3,0
3,2,1
0,4,0
1,7,1


In [69]:
frame.sort_values(by=['a', 'b'])

,b,a
2,-3,0
0,4,0
3,2,1
1,7,1


----

***Ranking***  assigns ranks from one through the number of valid data points in an array.
The rank methods for Series and DataFrame are the place to look; by default rank
breaks ties by assigning each group the mean rank:

In [71]:
obj = pd.Series([7, -5, 7, 4, 2, 0, 4])

In [72]:
obj.rank()

0    6.5
1    1.0
2    6.5
3    4.5
4    3.0
5    2.0
6    4.5
dtype: float64

Ranks can also be assigned according to the order in which they’re observed in the
data:

In [73]:
obj.rank(method='first')

0    6.0
1    1.0
2    7.0
3    4.0
4    3.0
5    2.0
6    5.0
dtype: float64

Here, instead of using the average rank 6.5 for the entries 0 and 2, they instead have
been set to 6 and 7 because label 0 precedes label 2 in the data.

You can rank in descending order, too:

In [74]:
# Assign tie values the maximum rank in the group
obj.rank(ascending=False, method='max')

0    2.0
1    7.0
2    2.0
3    4.0
4    5.0
5    6.0
6    4.0
dtype: float64

See Table BELOW for a list of tie-breaking methods available.

In [76]:
import pandas as pd
from IPython.display import display

# Create the table data
data = {
    "Method": ["average", "min", "max", "first", "dense"],
    "Description": [
        "Default: assign the average rank to each entry in the equal group",
        "Use the minimum rank for the whole group",
        "Use the maximum rank for the whole group",
        "Assign ranks in the order the values appear in the data",
        "Like method='min', but ranks always increase by 1 in between groups rather than the number of equal elements in a group"
    ]
}

df = pd.DataFrame(data)

# Apply styling for a clean documentation-style table
styled_table = (
    df.style
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "14px"),
                                    ("text-align", "center"),
                                    ("background-color", "#f2f2f2"),
                                    ("border", "1px solid black")]},
        {"selector": "td", "props": [("font-size", "13px"),
                                    ("border", "1px solid #999"),
                                    ("padding", "6px"),
                                    ("white-space", "normal")]},
        {"selector": "table", "props": [("border-collapse", "collapse"),
                                       ("width", "100%")]}
    ])
    .set_properties(**{
        "text-align": "left"
    })
)

display(styled_table)

,Method,Description
0,average,Default: assign the average rank to each entry in the equal group
1,min,Use the minimum rank for the whole group
2,max,Use the maximum rank for the whole group
3,first,Assign ranks in the order the values appear in the data
4,dense,"Like method='min', but ranks always increase by 1 in between groups rather than the number of equal elements in a group"


----

DataFrame can compute ranks over the rows or the columns:

In [77]:
frame = pd.DataFrame({'b': [4.3, 7, -3, 2], 'a': [0, 1, 0, 1],'c': [-2, 5, 8, -2.5]})

In [78]:
frame

,b,a,c
0,4.3,0,-2.0
1,7.0,1,5.0
2,-3.0,0,8.0
3,2.0,1,-2.5


In [79]:
frame.rank(axis='columns')

,b,a,c
0,3.0,2.0,1.0
1,3.0,1.0,2.0
2,1.0,2.0,3.0
3,3.0,2.0,1.0


### Axis Indexes with Duplicate Labels

Up until now all of the examples we’ve looked at have had unique axis labels (index
values). While many pandas functions (like reindex) require that the labels be
unique, it’s not mandatory. Let’s consider a small Series with duplicate indices:

In [80]:
obj = pd.Series(range(5), index=['a', 'a', 'b', 'b', 'c'])

In [81]:
obj

a    0
a    1
b    2
b    3
c    4
dtype: int64

The index’s `is_unique` property can tell you whether its labels are unique or not:

In [83]:
obj.index.is_unique

False

Data selection is one of the main things that behaves differently with duplicates.
Indexing a label with multiple entries returns a Series, while single entries return a
scalar value:

In [84]:
obj['a']

a    0
a    1
dtype: int64

In [85]:
obj['c']

np.int64(4)

This can make your code more complicated, as the output type from indexing can
vary based on whether a label is repeated or not.

The same logic extends to indexing rows in a DataFrame:

In [87]:
df = pd.DataFrame(np.random.randn(4, 3), index=['a', 'a', 'b', 'b'])

In [89]:
df 

,0,1,2
a,1.066684,0.558031,-0.176292
a,-0.450392,0.723040,0.171355
b,0.086872,-0.042987,-0.418593
b,-0.149787,0.367584,-0.345782


In [90]:
df.loc['b']

,0,1,2
b,0.086872,-0.042987,-0.418593
b,-0.149787,0.367584,-0.345782


## 5.3 Summarizing and Computing Descriptive Statistics

pandas objects are equipped with a set of common mathematical and statistical methods.
Most of these fall into the category of *reductions* or *summary statistics*, methods
that extract a single value (like the sum or mean) from a Series or a Series of values
from the rows or columns of a DataFrame. Compared with the similar methods
found on NumPy arrays, they have built-in handling for missing data. Consider a
small DataFrame:

In [13]:
df = pd.DataFrame([[1.4, np.nan], [7.1, -4.5],[np.nan, np.nan], [0.75, -1.3]],index=['a', 'b', 'c', 'd'],columns=['one', 'two'])

In [14]:
df

,one,two
a,1.40,NaN
b,7.10,-4.5
c,NaN,NaN
d,0.75,-1.3


Calling DataFrame’s sum method returns a Series containing column sums:

In [15]:
df.sum()

one    9.25
two   -5.80
dtype: float64

Passing axis='columns' or axis=1 sums across the columns instead:

In [16]:
df.sum(axis='columns')

a    1.40
b    2.60
c    0.00
d   -0.55
dtype: float64

NA values are excluded unless the entire slice (row or column in this case) is NA.
This can be disabled with the `skipna` option:

In [17]:
df.mean(axis='columns', skipna=False)

a      NaN
b    1.300
c      NaN
d   -0.275
dtype: float64

In [18]:
import pandas as pd
from IPython.display import display, HTML

# Table data
data = {
    "Method": ["axis", "skipna", "level"],
    "Description": [
        "Axis to reduce over; 0 for DataFrame’s rows and 1 for columns",
        "Exclude missing values; True by default",
        "Reduce grouped by level if the axis is hierarchically indexed (MultiIndex)"
    ]
}

data_df = pd.DataFrame(data)

# Styled table
styled_table = (
    data_df.style
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th",
         "props": [("background-color", "#f2f2f2"),
                   ("font-weight", "bold"),
                   ("text-align", "center"),
                   ("border", "1px solid #999"),
                   ("padding", "8px")]},

        {"selector": "td",
         "props": [("border", "1px solid #999"),
                   ("padding", "8px"),
                   ("white-space", "normal"),
                   ("text-align", "left")]},

        {"selector": "table",
         "props": [("border-collapse", "collapse"),
                   ("width", "100%")]}
    ])
)

# Display title and table together
display(HTML("<h3>Table : Options for Reduction Methods</h3>"))
display(styled_table)

Method,Description
axis,Axis to reduce over; 0 for DataFrame’s rows and 1 for columns
skipna,Exclude missing values; True by default
level,Reduce grouped by level if the axis is hierarchically indexed (MultiIndex)


Some methods, like `idxmin` and `idxmax`, return indirect statistics like the index value
where the minimum or maximum values are attained:

In [19]:
df.idxmax()

one    b
two    d
dtype: object

In [24]:
df.idxmax(axis= 'columns')

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_34292\2918599673.py:1: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  df.idxmax(axis= 'columns')


a    one
b    one
c    NaN
d    one
dtype: object

Other methods are accumulations:

In [20]:
df.cumsum()

,one,two
a,1.40,NaN
b,8.50,-4.5
c,NaN,NaN
d,9.25,-5.8


Another type of method is neither a reduction nor an accumulation. `describe` is one
such example, producing multiple summary statistics in one shot:

In [22]:
df.describe()

,one,two
count,3.000000,2.000000
mean,3.083333,-2.900000
std,3.493685,2.262742
min,0.750000,-4.500000
25%,1.075000,-3.700000
50%,1.400000,-2.900000
75%,4.250000,-2.100000
max,7.100000,-1.300000


On non-numeric data, `describe` produces alternative summary statistics:

In [25]:
obj = pd.Series(['a', 'a', 'b', 'c'] * 4)

In [26]:
obj

0     a
1     a
2     b
3     c
4     a
5     a
6     b
7     c
8     a
9     a
10    b
11    c
12    a
13    a
14    b
15    c
dtype: object

In [27]:
obj.describe()

count     16
unique     3
top        a
freq       8
dtype: object

In [29]:
import pandas as pd
from IPython.display import display, HTML

# Table data
statistics_table = pd.DataFrame({
    "Method": [
        "count",
        "describe",
        "min, max",
        "argmin, argmax",
        "idxmin, idxmax",
        "quantile",
        "sum",
        "mean",
        "median",
        "mad",
        "prod",
        "var",
        "std",
        "skew",
        "kurt",
        "cumsum",
        "cummin, cummax",
        "cumprod",
        "diff",
        "pct_change"
    ],
    "Description": [
        "Number of non-NA values",
        "Compute set of summary statistics for Series or each DataFrame column",
        "Compute minimum and maximum values",
        "Compute index locations (integers) at which minimum or maximum value obtained, respectively",
        "Compute index labels at which minimum or maximum value obtained, respectively",
        "Compute sample quantile ranging from 0 to 1",
        "Sum of values",
        "Mean of values",
        "Arithmetic median (50% quantile) of values",
        "Mean absolute deviation from mean value",
        "Product of all values",
        "Sample variance of values",
        "Sample standard deviation of values",
        "Sample skewness (third moment) of values",
        "Sample kurtosis (fourth moment) of values",
        "Cumulative sum of values",
        "Cumulative minimum or maximum of values, respectively",
        "Cumulative product of values",
        "Compute first arithmetic difference (useful for time series)",
        "Compute percent changes"
    ]
})

# Style the table
styled_statistics_table = (
    statistics_table.style
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th",
         "props": [("background-color", "#f2f2f2"),
                   ("font-weight", "bold"),
                   ("text-align", "center"),
                   ("border", "1px solid #999"),
                   ("padding", "8px")]},
        {"selector": "td",
         "props": [("border", "1px solid #999"),
                   ("padding", "8px"),
                   ("white-space", "normal"),
                   ("text-align", "left")]},
        {"selector": "table",
         "props": [("border-collapse", "collapse"),
                   ("width", "100%")]}
    ])
)

# Display title and table
display(HTML("<h3>Table: Descriptive and Summary Statistics</h3>"))
display(styled_statistics_table)

Method,Description
count,Number of non-NA values
describe,Compute set of summary statistics for Series or each DataFrame column
"min, max",Compute minimum and maximum values
"argmin, argmax","Compute index locations (integers) at which minimum or maximum value obtained, respectively"
"idxmin, idxmax","Compute index labels at which minimum or maximum value obtained, respectively"
quantile,Compute sample quantile ranging from 0 to 1
sum,Sum of values
mean,Mean of values
median,Arithmetic median (50% quantile) of values
mad,Mean absolute deviation from mean value


### Correlation and Covariance

Some summary statistics, like correlation and covariance, are computed from pairs of
arguments. Let’s consider some DataFrames of stock prices and volumes obtained
from Yahoo! Finance using the add-on pandas-datareader package. If you don’t
have it installed already, it can be obtained via conda or pip:

`conda install pandas-datareader`

I use the `pandas_datareader` module to download some data for a few stock tickers:

In [ ]:
import pandas_datareader.data as web

In [ ]:
price = pd.DataFrame({ticker: data['Adj Close']
                      for ticker, data in all_data.items()})

In [1]:
import sys
print(sys.executable)

C:\tools\Anaconda3\python.exe


In [7]:
!pip install yfinance

Defaulting to user installation because normal site-packages is not writeable


In [8]:
import site
site.addsitedir(r"C:\Users\212364780.HCAD\AppData\Roaming\Python\Python313\site-packages")


In [9]:
import yfinance as yf

In [12]:
import yfinance as yf
import pandas as pd

tickers = ['AAPL','IBM','MSFT','GOOG']

data = yf.download(tickers)

price = data['Close']
volume = data['Volume']

print(price.head())
print(volume.head())

[*********************100%***********************]  4 of 4 completed

Ticker            AAPL        GOOG         IBM        MSFT
Date                                                      
2026-02-09  274.619995  324.399994  294.660004  412.658142
2026-02-10  273.679993  318.630005  291.760010  412.328857
2026-02-11  275.500000  311.329987  272.809998  403.449127
2026-02-12  261.730011  309.369995  259.519989  400.924896
2026-02-13  255.779999  306.019989  262.380005  400.406097
Ticker          AAPL      GOOG       IBM      MSFT
Date                                              
2026-02-09  44623400  26103300   4627800  45480500
2026-02-10  34376900  25281900   3837300  44857900
2026-02-11  51931300  24008100   7628200  42491000
2026-02-12  81077200  28194000  12565400  40802400
2026-02-13  56290700  20236000   6842600  34091600


In [11]:
print(data.columns)

MultiIndex([( 'Close', 'AAPL'),
            ( 'Close', 'GOOG'),
            ( 'Close',  'IBM'),
            ( 'Close', 'MSFT'),
            (  'High', 'AAPL'),
            (  'High', 'GOOG'),
            (  'High',  'IBM'),
            (  'High', 'MSFT'),
            (   'Low', 'AAPL'),
            (   'Low', 'GOOG'),
            (   'Low',  'IBM'),
            (   'Low', 'MSFT'),
            (  'Open', 'AAPL'),
            (  'Open', 'GOOG'),
            (  'Open',  'IBM'),
            (  'Open', 'MSFT'),
            ('Volume', 'AAPL'),
            ('Volume', 'GOOG'),
            ('Volume',  'IBM'),
            ('Volume', 'MSFT')],
           names=['Price', 'Ticker'])


In [13]:
returns = price.pct_change()

In [14]:
returns.tail()

Ticker,AAPL,GOOG,IBM,MSFT
Date,,,,
2026-03-02,0.002044,-0.016280,-0.003497,0.014793
2026-03-03,-0.003664,-0.009140,0.024690,0.013499
2026-03-04,-0.004664,-0.000362,0.019488,0.003144
2026-03-05,-0.008495,-0.008370,0.025954,0.013524
2026-03-06,-0.010873,-0.008674,0.008965,-0.004188


The `corr` method of Series computes the correlation of the overlapping, non-NA,
aligned-by-index values in two Series. Relatedly, `cov` computes the covariance:

In [15]:
returns['MSFT'].corr(returns['IBM'])

np.float64(0.8025529869501011)

In [16]:
returns['MSFT'].cov(returns['IBM'])

np.float64(0.0004925099026847361)

DataFrame’s `corr` and `cov` methods, on the other hand, return a full correlation or
covariance matrix as a DataFrame, respectively:

In [17]:
returns.corr()

Ticker,AAPL,GOOG,IBM,MSFT
Ticker,,,,
AAPL,1.000000,0.043813,0.071639,0.143051
GOOG,0.043813,1.000000,0.233651,0.069709
IBM,0.071639,0.233651,1.000000,0.802553
MSFT,0.143051,0.069709,0.802553,1.000000


In [18]:
returns.cov()

Ticker,AAPL,GOOG,IBM,MSFT
Ticker,,,,
AAPL,0.000364,0.000012,0.000056,0.000041
GOOG,0.000012,0.000197,0.000133,0.000015
IBM,0.000056,0.000133,0.001658,0.000493
MSFT,0.000041,0.000015,0.000493,0.000227


Since MSFT is a valid Python attribute, we can also select these columns using more
concise syntax:

In [19]:
returns.MSFT.corr(returns.IBM)

np.float64(0.8025529869501011)

Using DataFrame’s `corrwith` method, you can compute pairwise correlations
between a DataFrame’s columns or rows with another Series or DataFrame. Passing a
Series returns a Series with the correlation value computed for each column:



In [20]:
returns.corrwith(returns.IBM)

Ticker
AAPL    0.071639
GOOG    0.233651
IBM     1.000000
MSFT    0.802553
dtype: float64

Passing a DataFrame computes the correlations of matching column names. Here I
compute correlations of percent changes with volume:

In [21]:
returns.corrwith(volume)

Ticker
AAPL   -0.493988
GOOG    0.397704
IBM    -0.557499
MSFT   -0.289227
dtype: float64

### Unique Values, Value Counts, and Membership

Another class of related methods extracts information about the values contained in a
one-dimensional Series. To illustrate these, consider this example:

In [22]:
obj = pd.Series(['c', 'a', 'd', 'a', 'a', 'b', 'b', 'c', 'c'])

The first function is `unique`, which gives you an array of the unique values in a Series:

In [24]:
uniques = obj.unique()

In [25]:
uniques 

array(['c', 'a', 'd', 'b'], dtype=object)

The unique values are not necessarily returned in sorted order, but could be sorted
after the fact if needed (`uniques.sort()`). Relatedly, `value_counts` computes a Series
containing value frequencies:

In [27]:
obj.value_counts()

c    3
a    3
b    2
d    1
Name: count, dtype: int64

In [29]:
uniques.sort()

In [30]:
uniques

array(['a', 'b', 'c', 'd'], dtype=object)

The Series is sorted by value in descending order as a convenience. `value_counts` is
also available as a top-level pandas method that can be used with any array or
sequence:

In [31]:
pd.value_counts(obj.values, sort=False)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_26596\3084372382.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(obj.values, sort=False)


c    3
a    3
d    1
b    2
Name: count, dtype: int64

`isin` performs a vectorized set membership check and can be useful in filtering a
dataset down to a subset of values in a Series or column in a DataFrame:

In [32]:
obj

0    c
1    a
2    d
3    a
4    a
5    b
6    b
7    c
8    c
dtype: object

In [33]:
mask = obj.isin(['b', 'c'])

In [34]:
mask

0     True
1    False
2    False
3    False
4    False
5     True
6     True
7     True
8     True
dtype: bool

In [35]:
obj[mask]

0    c
5    b
6    b
7    c
8    c
dtype: object

Related to `isin` is the `Index.get_indexer` method, which gives you an index array
from an array of possibly non-distinct values into another array of distinct values:

In [36]:
to_match = pd.Series(['c', 'a', 'b', 'b', 'c', 'a'])

In [37]:
unique_vals = pd.Series(['c', 'b', 'a'])

In [38]:
pd.Index(unique_vals).get_indexer(to_match)

array([0, 2, 1, 1, 0, 2])

In [40]:
import pandas as pd
from IPython.display import display, HTML

# Table data
series_methods_table = pd.DataFrame({
    "Method": [
        "isin",
        "match",
        "unique",
        "value_counts"
    ],
    "Description": [
        "Compute boolean array indicating whether each Series value is contained in the passed sequence of values",
        "Compute integer indices for each value in an array into another array of distinct values; helpful for data alignment and join-type operations",
        "Compute array of unique values in a Series, returned in the order observed",
        "Return a Series containing unique values as its index and the counts as values"
    ]
})

# Style for a clean documentation-style table
styled_table = (
    series_methods_table.style
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th",
         "props": [("background-color", "#f2f2f2"),
                   ("font-weight", "bold"),
                   ("text-align", "center"),
                   ("border", "1px solid #999"),
                   ("padding", "8px")]},
        {"selector": "td",
         "props": [("border", "1px solid #999"),
                   ("padding", "8px"),
                   ("white-space", "normal"),
                   ("text-align", "left")]},
        {"selector": "table",
         "props": [("border-collapse", "collapse"),
                   ("width", "100%")]}
    ])
)

# Display title and table
display(HTML("<h3>Table: Unique, Value Counts, and Set Membership Methods</h3>"))
display(styled_table)

Method,Description
isin,Compute boolean array indicating whether each Series value is contained in the passed sequence of values
match,Compute integer indices for each value in an array into another array of distinct values; helpful for data alignment and join-type operations
unique,"Compute array of unique values in a Series, returned in the order observed"
value_counts,Return a Series containing unique values as its index and the counts as values


In some cases, you may want to compute a histogram on multiple related columns in
a DataFrame. Here’s an example:

In [42]:
data = pd.DataFrame({'Qu1': [1, 3, 4, 3, 4],'Qu2': [2, 3, 1, 2, 3],'Qu3': [1, 5, 2, 4, 4]})

In [43]:
data

,Qu1,Qu2,Qu3
0,1,2,1
1,3,3,5
2,4,1,2
3,3,2,4
4,4,3,4


In [44]:
result = data.apply(pd.value_counts).fillna(0)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_26596\2451317050.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  result = data.apply(pd.value_counts).fillna(0)


In [45]:
result

,Qu1,Qu2,Qu3
1,1.0,1.0,1.0
2,0.0,2.0,1.0
3,2.0,2.0,0.0
4,2.0,0.0,2.0
5,0.0,0.0,1.0


Let’s break down the line step-by-step because the key idea is how **`DataFrame.apply()` interacts with `pd.value_counts`**.

```python
data = pd.DataFrame({
    'Qu1': [1, 3, 4, 3, 4],
    'Qu2': [2, 3, 1, 2, 3],
    'Qu3': [1, 5, 2, 4, 4]
})

result = data.apply(pd.value_counts).fillna(0)
```

---

# 1️⃣ What `apply()` actually does here

By default:

```python
DataFrame.apply(func)
```

is equivalent to

```python
DataFrame.apply(func, axis=0)
```

That means:

👉 **The function is applied column by column.**

So **`pd.value_counts` does NOT receive the whole DataFrame.**

Instead it receives **one Series at a time**.

Execution internally behaves like:

```python
pd.value_counts(data['Qu1'])
pd.value_counts(data['Qu2'])
pd.value_counts(data['Qu3'])
```

---

# 2️⃣ What `pd.value_counts()` expects as input

`pd.value_counts()` expects:

```
Series or 1-D array-like
```

Example:

```python
pd.value_counts(data['Qu1'])
```

Input:

```
[1, 3, 4, 3, 4]
```

Output:

```
3    2
4    2
1    1
dtype: int64
```

Meaning:

| value | count |
| ----- | ----- |
| 3     | 2     |
| 4     | 2     |
| 1     | 1     |

---

# 3️⃣ What happens for each column

### Column Qu1

```
[1,3,4,3,4]
```

```
1 → 1
3 → 2
4 → 2
```

---

### Column Qu2

```
[2,3,1,2,3]
```

```
1 → 1
2 → 2
3 → 2
```

---

### Column Qu3

```
[1,5,2,4,4]
```

```
1 → 1
2 → 1
4 → 2
5 → 1
```

---

# 4️⃣ How `apply()` combines the results

`apply()` aligns results by the **index (unique values)**.

Combined result becomes:

| value | Qu1 | Qu2 | Qu3 |
| ----- | --- | --- | --- |
| 1     | 1   | 1   | 1   |
| 2     | NaN | 2   | 1   |
| 3     | 2   | 2   | NaN |
| 4     | 2   | NaN | 2   |
| 5     | NaN | NaN | 1   |

---

# 5️⃣ Why `.fillna(0)` is used

Some values don't exist in certain columns.

Example:

* value **5** exists only in **Qu3**
* value **2** doesn't exist in **Qu1**

These become `NaN`.

```
.fillna(0)
```

replaces them with **0**.

---

# 6️⃣ Final result

| value | Qu1 | Qu2 | Qu3 |
| ----- | --- | --- | --- |
| 1     | 1   | 1   | 1   |
| 2     | 0   | 2   | 1   |
| 3     | 2   | 2   | 0   |
| 4     | 2   | 0   | 2   |
| 5     | 0   | 0   | 1   |

This table shows:

👉 **How many times each value appears in each column**

---

# 7️⃣ Mental model (important)

Think of this as:

```
Frequency table for each column
```

or

```
Histogram of values per column
```

---

# 8️⃣ Equivalent manual code (for clarity)

This is what pandas essentially does internally:

```python
result = pd.concat(
    [pd.value_counts(data[col]) for col in data.columns],
    axis=1
)
```

Then:

```
fillna(0)
```

---

✅ **Key takeaway**

| Function            | Receives                    |
| ------------------- | --------------------------- |
| `DataFrame.apply()` | each column as a **Series** |
| `pd.value_counts()` | that **Series**             |
| Return              | counts of unique values     |

---

💡 If you'd like, I can also show a **visual step-by-step diagram of how `apply()` executes internally**, which makes understanding pandas transformations **10× easier when learning data science**.


## 5.4 Conclusion

In the next chapter, we’ll discuss tools for reading (or loading) and writing datasets
with pandas. After that, we’ll dig deeper into data cleaning, wrangling, analysis, and
visualization tools using pandas.

https://data.ny.gov/Transportation/MTA-Bus-Customer-Journey-Focused-Metrics-Beginning/8mkn-d32t/data_preview
Download this xml to a local folder

In [47]:
from lxml import objectify

In [48]:
path = 'examples/Performance_MNR.xml'